# Qwen3-14B

Settings: Accelerator GPU T4 x2, Internet on. The model is 27.5 GB, split across
both cards with a small tail in host memory. T4 has no usable bf16, so float16.

Two stages, each with its own archive, so the first is safe before the second starts.
Stage one is the baseline and both formats on the weights, which is what the task
statement asks for. Stage two is the method from this project, whose Hessian solve at
this size is the most expensive step in the whole project.

Two benchmark tasks, full sets, no subsampling. One benchmark pass here costs about
sixteen times what it costs on Qwen3-4B, because part of the model crosses the bus on
every forward.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'  # less fragmentation
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.device_count(), 'GPUs')

In [ ]:
!pip -q install -U 'transformers>=4.51' safetensors accelerate datasets zstandard 'lm-eval>=0.4.5' 2>&1 | tail -3

In [ ]:
import os
os.chdir('/kaggle/working')
!git clone -q --depth 1 https://github.com/microsoft/microxcaling.git vendor/microxcaling
!mkdir -p src scripts results

## Project sources

In [ ]:
%%writefile src/compress.py
# rate distortion coarsening of MX codes plus entropy coding of codes and scales
import io
import os

import numpy as np
import torch
import zstandard as zstd

from mxfmt import sorted_levels, level_table, ELEM_BITS, SCALE_BITS, BLOCK

ZLEVEL = int(os.environ.get("MX_ZSTD_LEVEL", 12))


def pack_codes(codes, bits):
    """Pack integer codes into bytes, two per byte for 4 bit formats."""
    a = codes.astype(np.uint8).ravel()
    if bits == 4:
        if a.size % 2:
            a = np.append(a, 0)
        return ((a[0::2] << 4) | a[1::2]).tobytes()
    return a.tobytes()


def zcompress(b, level=ZLEVEL):
    return zstd.ZstdCompressor(level=level).compress(b)


def zdecompress(b, size):
    return zstd.ZstdDecompressor().decompress(b, max_output_size=size)


def unpack_codes(raw, n, bits):
    """Inverse of pack_codes."""
    a = np.frombuffer(raw, dtype=np.uint8)
    if bits == 4:
        out = np.empty(a.size * 2, dtype=np.uint8)
        out[0::2] = a >> 4
        out[1::2] = a & 0x0F
        return out[:n]
    return a[:n].copy()


def entropy(counts):
    p = counts[counts > 0].astype(np.float64)
    p /= p.sum()
    return float(-(p * np.log2(p)).sum())


def encode_scales(shared_exp):
    """Delta code E8M0 exponents along the block axis, then zstd."""
    e = shared_exp.numpy().astype(np.int16)
    d = np.diff(e, axis=-1, prepend=np.zeros_like(e[..., :1]))
    z = ((d << 1) ^ (d >> 15)).astype(np.uint16)  # zigzag keeps small deltas small
    narrow = z.max() < 256
    raw = z.astype(np.uint8).tobytes() if narrow else z.tobytes()
    return zcompress(raw), e.size, (narrow, e.shape)


def decode_scales(blob, meta):
    """Inverse of encode_scales."""
    narrow, shape = meta
    n = int(np.prod(shape))
    raw = zdecompress(blob, n * (1 if narrow else 2))
    z = np.frombuffer(raw, dtype=np.uint8 if narrow else np.uint16).astype(np.uint16)
    d = ((z >> 1).astype(np.int16) ^ -(z & 1).astype(np.int16)).reshape(shape)
    return torch.from_numpy(np.cumsum(d, axis=-1).astype(np.int16))


def rank_of_code(codes, elem_format):
    """Map format bit codes to indices in the value sorted level list."""
    _, codes_of_level = sorted_levels(elem_format)
    lut = np.zeros(256, dtype=np.uint8)
    lut[codes_of_level] = np.arange(len(codes_of_level), dtype=np.uint8)
    return lut[codes]


def code_of_rank(ranks, elem_format):
    _, codes_of_level = sorted_levels(elem_format)
    return codes_of_level[ranks]


def rd_coarsen(ranks, levels, w, _unused, lam, protect, rate):
    """Move each code at most one level if the rate gain outweighs the weighted distortion."""
    n_lv = len(levels)
    best = ranks.copy()
    cur_cost = lam * rate[ranks]
    for off in (-1, 1):
        cand = ranks.astype(np.int16) + off
        ok = (cand >= 0) & (cand < n_lv) & (~protect)
        cand_c = np.clip(cand, 0, n_lv - 1)
        d = (levels[cand_c] - levels[ranks]) * w
        cost = d * d + lam * rate[cand_c]
        take = ok & (cost < cur_cost)
        best = np.where(take, cand_c, best).astype(np.uint8)
        cur_cost = np.where(take, cost, cur_cost)
    return best


def _importance(shape, act_norm, block):
    """Broadcast per input channel activation norms onto the blocked tensor shape."""
    if act_norm is None:
        return None
    a = act_norm.numpy().astype(np.float32)
    pad = shape[-2] * block - len(a)
    if pad > 0:
        a = np.concatenate([a, np.zeros(pad, dtype=np.float32)])
    return a.reshape(shape[-2], block)


def compress_tensor(codes, shared_exp, elem_format, act_norm=None, alpha=0.0, ref_mse=None,
                    protect_top=0.01, block=BLOCK, chunk_elems=8_000_000):
    """Entropy code one quantized tensor, optionally coarsening the unimportant part first."""
    bits = ELEM_BITS[elem_format]
    levels, codes_of_level = sorted_levels(elem_format)
    levels = levels.astype(np.float32)
    ranks = rank_of_code(codes.numpy(), elem_format)
    shape = codes.shape
    n = int(np.prod(shape))
    exps = shared_exp.numpy().astype(np.float32)
    imp2d = _importance(shape, act_norm, block)

    lam = 0.0 if alpha <= 0 else alpha * float(ref_mse)
    hist = np.bincount(ranks.ravel(), minlength=len(levels)).astype(np.float64)
    rate = -np.log2(np.maximum(hist / hist.sum(), 1e-12))

    thr = np.inf
    if lam > 0 and protect_top > 0:
        sel = np.random.default_rng(0).integers(0, shape[0], size=min(shape[0], 4096))
        sv = np.abs(levels[ranks[sel]]) * (2.0 ** exps[sel])[..., None]
        if imp2d is not None:
            sv = sv * imp2d
        thr = float(np.quantile(sv, 1 - protect_top))
        del sv

    rows_per_chunk = max(1, chunk_elems // max(1, n // shape[0]))
    buf = io.BytesIO()
    writer = zstd.ZstdCompressor(level=ZLEVEL).stream_writer(buf)
    changed, sq_dw, new_hist = 0, 0.0, np.zeros(len(levels), dtype=np.int64)
    out_chunks = []

    for i in range(0, shape[0], rows_per_chunk):
        r = ranks[i:i + rows_per_chunk]
        sc = (2.0 ** exps[i:i + rows_per_chunk])[..., None]
        w = sc if imp2d is None else sc * imp2d
        if lam > 0:
            val = np.abs(levels[r]) * w
            nr = rd_coarsen(r, levels, w, np.ones(1, dtype=np.float32), lam, val >= thr, rate)
            del val
        else:
            nr = r
        d = (levels[nr] - levels[r]) * sc
        sq_dw += float((d.astype(np.float64) ** 2).sum())
        changed += int((nr != r).sum())
        new_hist += np.bincount(nr.ravel(), minlength=len(levels))
        oc = codes_of_level[nr]
        out_chunks.append(oc)
        writer.write(pack_codes(oc, bits))
        del r, sc, w, d, nr

    writer.flush(zstd.FLUSH_FRAME)
    cbytes = buf.tell()
    sbytes, n_blocks, smeta = encode_scales(shared_exp)

    return dict(
        codes=torch.from_numpy(np.concatenate(out_chunks, axis=0)),
        blob_codes=buf.getvalue(), blob_scales=sbytes, scale_meta=smeta, elem_format=elem_format,
        shape=tuple(shape),
        bytes_codes=cbytes, bytes_scales=len(sbytes), n=n, n_blocks=n_blocks,
        bits_per_value=(cbytes + len(sbytes)) * 8 / n,
        bits_codes=cbytes * 8 / n, bits_scales=len(sbytes) * 8 / n,
        code_entropy=entropy(new_hist), changed_frac=changed / n, alpha=alpha, lam=lam,
        dw_rms=float(np.sqrt(sq_dw / n)),
    )


def decompress_tensor(r):
    """Rebuild codes and exponents from the compressed blobs of compress_tensor."""
    bits = ELEM_BITS[r["elem_format"]]
    shape = r["shape"]
    n = int(np.prod(shape))
    raw = zdecompress(r["blob_codes"], (n + 1) // 2 if bits == 4 else n)
    codes = unpack_codes(raw, n, bits).reshape(shape)
    return torch.from_numpy(codes), decode_scales(r["blob_scales"], r["scale_meta"])

In [ ]:
%%writefile src/errcomp.py
# error compensated MX quantization, group size follows the MX block
import torch

from mxfmt import BLOCK, block_exponent, quantize_with_exponent


class Hessian:
    """Accumulates X^T X over calibration batches for one linear layer."""

    def __init__(self, n_in, device="cpu"):
        self.h = torch.zeros(n_in, n_in, dtype=torch.float32, device=device)
        self.n = 0

    def add(self, x):
        x = x.reshape(-1, x.shape[-1]).float()
        self.h += x.T @ x
        self.n += x.shape[0]

    def finish(self):
        return self.h / max(self.n, 1)


@torch.no_grad()
def quantize_compensated(w, elem_format, h, block=BLOCK, damp=0.01, search=False):
    """Quantize a weight block by block, pushing each block error onto the columns still to come."""
    w = w.float().clone()
    n_in = w.shape[1]
    d = torch.arange(n_in, device=w.device)
    diag = torch.diag(h)
    dead = diag == 0
    h = h.clone()
    h[dead, dead] = 1
    w[:, dead] = 0
    h[d, d] += damp * diag.mean()

    try:
        hinv = torch.linalg.cholesky(h)
        hinv = torch.cholesky_inverse(hinv)
        hinv = torch.linalg.cholesky(hinv, upper=True)
    except RuntimeError:
        return quantize_with_exponent(w, elem_format, block_exponent(
            w.reshape(w.shape[0], -1, block), elem_format).unsqueeze(-1).expand(-1, -1, block).reshape(w.shape))

    q = torch.zeros_like(w)
    for i1 in range(0, n_in, block):
        i2 = min(i1 + block, n_in)
        cnt = i2 - i1
        wb = w[:, i1:i2].clone()
        qb = torch.zeros_like(wb)
        eb = torch.zeros_like(wb)
        hb = hinv[i1:i2, i1:i2]

        exp = block_exponent(wb, elem_format).unsqueeze(-1)
        if search:
            best = None
            for off in (0, 1):
                cand = quantize_with_exponent(wb, elem_format, exp + off)
                err = (wb - cand).pow(2).sum(-1, keepdim=True)
                if best is None or True:
                    if best is None:
                        best, best_err, best_exp = cand, err, exp + off
                    else:
                        take = err < best_err
                        best = torch.where(take, cand, best)
                        best_exp = torch.where(take, exp + off, best_exp)
                        best_err = torch.where(take, err, best_err)
            exp = best_exp

        for j in range(cnt):
            col = wb[:, j]
            qc = quantize_with_exponent(col.unsqueeze(-1), elem_format, exp).squeeze(-1)
            qb[:, j] = qc
            e = (col - qc) / hb[j, j]
            if j + 1 < cnt:
                wb[:, j + 1:] -= e.unsqueeze(1) * hb[j, j + 1:].unsqueeze(0)
            eb[:, j] = e

        q[:, i1:i2] = qb
        if i2 < n_in:
            w[:, i2:] -= eb @ hinv[i1:i2, i2:]

    return q


def real_weight(m):
    """Offloaded modules hold a meta placeholder, the data lives in the accelerate hook."""
    w = m.weight.data
    if not w.is_meta:
        return w
    wm = getattr(getattr(m, "_hf_hook", None), "weights_map", None)
    if wm is None:
        raise RuntimeError(
            "weight is on the meta device and no accelerate hook carries its data. "
            "Error compensation needs the real weights, so run it on a device that "
            "fits the model without offload.")
    return wm["weight"]


def _run(w, h, ef, damp, search):
    """Solve on the weight's own GPU when it fits, fall back to cpu on an allocation failure."""
    if w.device.type == "cuda":
        try:
            return quantize_compensated(w.float(), ef, h.to(w.device), damp=damp, search=search)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
    return quantize_compensated(w.float().cpu(), ef, h.cpu(), damp=damp, search=search)


def _layer_index(name):
    import re
    m = re.search(r"layers\.(\d+)\.", name)
    return int(m.group(1)) if m else -1


@torch.no_grad()
def apply_compensated(model, calib_ids, fmt, device, group=2, nsamples=16, seqlen=1024,
               damp=0.01, search=False, verbose=True):
    """Quantize all linears layer group by layer group so later groups see quantized inputs.

    Group size trades passes for memory: one Hessian is d_in by d_in floats, and down_proj on a
    4B model is already 378 MB of that.
    """
    import time
    from fakequant import target_linears
    from mxfmt import FORMATS

    ef = FORMATS[fmt]
    mods = target_linears(model)
    offloaded = [n for n, m in mods if m.weight.data.is_meta
                 and getattr(getattr(m, "_hf_hook", None), "weights_map", None) is None]
    if offloaded:
        raise RuntimeError(
            f"{len(offloaded)} of {len(mods)} weights are on the meta device with no data behind "
            f"them, first is {offloaded[0]}. Error compensation reads every weight directly, so "
            f"it needs the model to fit without offload. Check this before the run rather than "
            f"after, it costs hours on a large model.")
    layers = sorted({_layer_index(n) for n, _ in mods if _layer_index(n) >= 0})
    t0 = time.time()

    for start in range(0, len(layers), group):
        sel = set(layers[start:start + group])
        todo = [(n, m) for n, m in mods if _layer_index(n) in sel]
        if not todo:
            continue
        hs, handles = {}, []

        def mk(name, n_in):
            hs[name] = Hessian(n_in)

            def hook(mod, args):
                hs[name].add(args[0].detach().cpu())
            return hook

        for n, m in todo:
            handles.append(m.register_forward_pre_hook(mk(n, m.in_features)))

        step = calib_ids.numel() // nsamples
        for i in range(nsamples):
            x = calib_ids[i * step:i * step + seqlen].unsqueeze(0).to(device)
            if x.shape[1] < seqlen:
                break
            model(x)
        for h in handles:
            h.remove()

        for n, m in todo:
            w = real_weight(m)
            h = hs[n].finish()
            q = _run(w, h, ef, damp, search)
            m.weight.data = q.to(device=w.device, dtype=w.dtype)
            del hs[n], h
        if verbose:
            print(f"  layers {sorted(sel)} done, {time.time()-t0:.0f}s", flush=True)
    return len(mods)


def calib_tokens(tok, source="wikitext", n_chars=2_000_000):
    """Calibration text, either wikitext or a slice of C4 web text."""
    from datasets import load_dataset
    if source == "c4":
        ds = load_dataset("allenai/c4", data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
                          split="train", streaming=True)
        parts, total = [], 0
        for r in ds:
            parts.append(r["text"])
            total += len(r["text"])
            if total >= n_chars:
                break
        text = "\n\n".join(parts)
    else:
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        text = "\n\n".join(ds["text"][:20000])
    return tok(text, return_tensors="pt").input_ids[0]

In [ ]:
%%writefile src/fakequant.py
# in place MX fake quantization of a loaded HF model, weights and activations
import json

import torch
import torch.nn as nn

from mxfmt import quantize, quantize_search, FORMATS

SKIP = ("lm_head",)


def target_linears(model, include_lm_head=False):
    """All nn.Linear modules that carry transformer matmul weights."""
    out = []
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            if not include_lm_head and any(s in name for s in SKIP):
                continue
            out.append((name, m))
    return out


@torch.no_grad()
def quantize_weights(model, fmt, include_lm_head=False, chunk_rows=2048, search=False,
                     act_norms=None, offsets=(0, 1)):
    """Replace each linear weight by its MX round trip, in row chunks to cap memory."""
    ef = FORMATS[fmt]
    n = 0
    for name, m in target_linears(model, include_lm_head):
        w = m.weight.data
        a = None if act_norms is None else act_norms.get(name)
        for i in range(0, w.shape[0], chunk_rows):
            c = w[i:i + chunk_rows].float()
            if search:
                q, _, _ = quantize_search(c, ef, offsets=offsets, act_norm=a)
            else:
                q = quantize(c, ef)
            w[i:i + chunk_rows] = q.to(w.dtype)
        n += w.numel()
    return n


class ActQuant:
    """Forward pre hooks that quantize linear inputs along the reduction axis."""

    def __init__(self, model, fmt, include_lm_head=False):
        self.ef = FORMATS[fmt]
        self.handles = []
        for name, m in target_linears(model, include_lm_head):
            self.handles.append(m.register_forward_pre_hook(self._hook))

    def _hook(self, module, args):
        x = args[0]
        return (quantize(x.float(), self.ef).to(x.dtype),) + args[1:]

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles = []


@torch.no_grad()
def apply_plan(model, plan, act_norms=None, protect_top=0.01, chunk_rows=4096):
    """Rebuild each linear weight from its planned format and coarsening strength."""
    from mxfmt import quantize_full, dequantize_codes
    from compress import compress_tensor

    total_bytes, total_n, touched = 0, 0, 0
    for name, m in target_linears(model):
        spec = plan.get(name + ".weight")
        if spec is None:
            continue
        ef = FORMATS[spec["fmt"]]
        a = None if act_norms is None else act_norms.get(name)
        w = m.weight.data
        for i in range(0, w.shape[0], chunk_rows):
            c = w[i:i + chunk_rows].float().cpu()  # packing and zstd run on cpu anyway
            deq, se, codes, meta = quantize_full(c, ef)
            if spec["alpha"] > 0:
                ref = (c - deq).pow(2).mean().item()
                r = compress_tensor(codes, se, ef, act_norm=a, alpha=spec["alpha"],
                                    ref_mse=ref, protect_top=protect_top)
                rec = dequantize_codes(r["codes"], se, ef, meta)
                total_bytes += r["bytes_codes"] + r["bytes_scales"]
            else:
                r = compress_tensor(codes, se, ef, alpha=0.0)
                rec = deq
                total_bytes += r["bytes_codes"] + r["bytes_scales"]
            w[i:i + chunk_rows] = rec.to(device=w.device, dtype=w.dtype)
            total_n += c.numel()
        touched += 1
    return dict(tensors=touched, params=total_n, bytes=total_bytes,
                bits_per_value=total_bytes * 8 / max(total_n, 1))


def build_model(model_id, weights="none", acts="none", device="cpu", dtype=torch.bfloat16,
                lm_head=False, plan_file=None, budget=None, calib=None, search=False,
                device_map=None, max_memory=None):
    """Load a model and apply one quantization configuration to it."""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(device_map=device_map) if device_map else {}
    if max_memory:
        kw["max_memory"] = {(int(k) if k.isdigit() else k): v
                            for k, v in (q.split("=") for q in max_memory.split(","))}
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, **kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, **kw)
    if not device_map:
        model = model.to(device)
    model = model.eval()

    info = {}
    if plan_file:
        with open(plan_file) as f:
            plans = json.load(f)
        norms = torch.load(calib) if calib else None
        info = apply_plan(model, plans["budgets"][budget]["plan"], norms)
    elif weights != "none":
        norms = torch.load(calib) if calib else None
        quantize_weights(model, weights, lm_head, search=search, act_norms=norms)
    act = ActQuant(model, acts, lm_head) if acts != "none" else None
    return model, tok, act, info

In [ ]:
%%writefile src/modelio.py
# streaming access to safetensors checkpoints, never loads a whole model
import glob, json, os, re
from safetensors import safe_open
from huggingface_hub import snapshot_download

PROJ_RE = re.compile(r"\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)\.weight$")
LAYER_RE = re.compile(r"layers\.(\d+)\.")


def local_path(model_id):
    return snapshot_download(model_id, allow_patterns=["*.json", "*.safetensors", "*.txt"])


def shard_files(path):
    return sorted(glob.glob(os.path.join(path, "*.safetensors")))


def iter_tensors(path, only_2d=True):
    """Yield (name, tensor) one at a time so peak memory stays at one tensor."""
    for f in shard_files(path):
        with safe_open(f, framework="pt") as sf:
            for name in sf.keys():
                t = sf.get_tensor(name)
                if only_2d and t.ndim != 2:
                    continue
                yield name, t
                del t


def tensor_index(path):
    """Map of tensor name to shape and dtype without reading the data."""
    out = {}
    for f in shard_files(path):
        with safe_open(f, framework="pt") as sf:
            for name in sf.keys():
                sl = sf.get_slice(name)
                out[name] = (tuple(sl.get_shape()), sl.get_dtype())
    return out


def classify(name):
    """Coarse role of a tensor, used to group the analysis."""
    m = PROJ_RE.search(name)
    if m:
        return m.group(1)
    if "embed_tokens" in name:
        return "embed_tokens"
    if "lm_head" in name:
        return "lm_head"
    if "norm" in name:
        return "norm"
    return "other"


def layer_of(name):
    m = LAYER_RE.search(name)
    return int(m.group(1)) if m else -1

In [ ]:
%%writefile src/mxfmt.py
# MX format helpers on top of microsoft/microxcaling
import sys, os
import numpy as np
import torch

_VENDOR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "vendor", "microxcaling")
if _VENDOR not in sys.path:
    sys.path.insert(0, _VENDOR)

from mx.mx_ops import _quantize_mx, _shared_exponents, _reshape_to_blocks, _undo_reshape_to_blocks
from mx.formats import ElemFormat, _get_format_params

BLOCK = 32
SCALE_BITS = 8

FORMATS = {"mxfp4": "fp4_e2m1", "mxfp8": "fp8_e4m3", "mxfp8_e5m2": "fp8_e5m2"}
ELEM_BITS = {"fp4_e2m1": 4, "fp8_e4m3": 8, "fp8_e5m2": 8}


def _level_table(elem_format):
    """All representable values of an element format, indexed by its bit code."""
    ebits, mbits, emax, max_norm, _ = _get_format_params(elem_format)
    man_bits = mbits - 2  # mbits counts sign and implicit one
    vals = np.zeros(2 ** (1 + ebits + man_bits), dtype=np.float64)
    bias = 2 ** (ebits - 1) - 1
    for s in (0, 1):
        for e in range(2 ** ebits):
            for m in range(2 ** man_bits):
                code = (s << (ebits + man_bits)) | (e << man_bits) | m
                if e == 0:
                    v = 2.0 ** (1 - bias) * (m / 2 ** man_bits)
                else:
                    v = 2.0 ** (e - bias) * (1 + m / 2 ** man_bits)
                if v > max_norm:
                    v = np.nan
                vals[code] = -v if s else v
    return vals


_TABLES = {}


def level_table(elem_format):
    if elem_format not in _TABLES:
        _TABLES[elem_format] = _level_table(elem_format)
    return _TABLES[elem_format]


def sorted_levels(elem_format):
    """Finite levels sorted ascending, with the code of each."""
    t = level_table(elem_format)
    codes = np.flatnonzero(np.isfinite(t))
    order = np.argsort(t[codes], kind="stable")
    return t[codes][order], codes[order].astype(np.uint8)


def quantize(w, elem_format, axis=-1, block=BLOCK):
    """Fake-quantize a tensor to an MX format. Returns the dequantized tensor."""
    return _quantize_mx(w, SCALE_BITS, elem_format, axes=[axis], block_size=block, round="nearest")


def quantize_full(w, elem_format, axis=-1, block=BLOCK):
    """Quantize and also return per-block E8M0 exponents and per-element integer codes."""
    w = w.float()
    deq = quantize(w, elem_format, axis, block)
    _, _, emax, _, _ = _get_format_params(elem_format)

    blocked, axes, orig_shape, padded_shape = _reshape_to_blocks(w, [axis], block)
    sax = axes[0] + 1
    shared_exp = _shared_exponents(blocked, method="max", axes=[sax], ebits=0) - emax
    lim = 2 ** (SCALE_BITS - 1) - 1
    shared_exp = shared_exp.clamp(-lim, lim)

    dq_blocked, _, _, _ = _reshape_to_blocks(deq, [axis], block)
    elems = dq_blocked / (2.0 ** shared_exp)

    levels, codes_of_level = sorted_levels(elem_format)
    lv = torch.tensor(levels, dtype=torch.float32, device=w.device)
    idx = torch.searchsorted(lv, elems.contiguous().flatten().clamp(lv[0], lv[-1]))
    idx = idx.clamp(1, len(lv) - 1)
    lo, hi = lv[idx - 1], lv[idx]
    pick = torch.where((elems.flatten() - lo).abs() <= (hi - elems.flatten()).abs(), idx - 1, idx)
    codes = torch.from_numpy(codes_of_level.astype(np.int16)).to(w.device)[pick].reshape(dq_blocked.shape)

    return deq, shared_exp.squeeze(sax).to(torch.int16), codes.to(torch.uint8), (axes, orig_shape, padded_shape)


def dequantize_codes(codes, shared_exp, elem_format, meta):
    """Inverse of quantize_full, used to check round trips and to rebuild edited tensors."""
    axes, orig_shape, padded_shape = meta
    t = torch.tensor(level_table(elem_format), dtype=torch.float32, device=codes.device)
    vals = t[codes.long()] * (2.0 ** shared_exp.float().to(codes.device).unsqueeze(axes[0] + 1))
    return _undo_reshape_to_blocks(vals, padded_shape, orig_shape, axes)


def bits_per_value(elem_format, block=BLOCK):
    """Storage cost of one quantized value including its share of the block scale."""
    return ELEM_BITS[elem_format] + SCALE_BITS / block


def block_max_ratio(w, elem_format, axis=-1, block=BLOCK):
    """Per block max|x|/scale, values above max_norm mean the block max is saturated."""
    _, _, emax, max_norm, _ = _get_format_params(elem_format)
    blocked, axes, _, _ = _reshape_to_blocks(w.float(), [axis], block)
    sax = axes[0] + 1
    shared_exp = (_shared_exponents(blocked, method="max", axes=[sax], ebits=0) - emax).clamp(-127, 127)
    return blocked.abs() / (2.0 ** shared_exp), max_norm


def block_shape_stats(w, axis=-1, block=BLOCK):
    """Kurtosis measured inside each block, averaged over blocks, and the block max to rms ratio."""
    blocked, _, _, _ = _reshape_to_blocks(w.float(), [axis], block)
    m2 = blocked.pow(2).mean(-1)
    m4 = blocked.pow(4).mean(-1)
    ok = m2 > 0
    kurt = torch.where(ok, m4 / m2.clamp_min(1e-30) ** 2, torch.full_like(m2, float("nan")))
    crest = torch.where(ok, blocked.abs().amax(-1) / m2.clamp_min(1e-30).sqrt(), torch.full_like(m2, float("nan")))
    return kurt, crest


def quantize_search(w, elem_format, axis=-1, block=BLOCK, offsets=(0, 1), act_norm=None, ref=None):
    """Pick the block exponent that minimises weighted block error instead of using the floor rule."""
    from mx.elemwise_ops import _quantize_elemwise_core
    ebits, mbits, emax, max_norm, _ = _get_format_params(elem_format)
    blocked, axes, orig_shape, padded_shape = _reshape_to_blocks(w.float(), [axis], block)
    sax = axes[0] + 1
    base = (_shared_exponents(blocked, method="max", axes=[sax], ebits=0) - emax).clamp(-127, 127)

    if act_norm is not None:
        a = act_norm.float().to(w.device)
        pad = padded_shape[axes[0]] - a.numel()
        if pad > 0:
            a = torch.cat([a, torch.zeros(pad, device=a.device)])
        wgt = a.reshape(1, -1, block).unsqueeze(0)[0] if blocked.dim() == 3 else a.reshape(*blocked.shape[1:])
        wgt = wgt.expand_as(blocked) ** 2
    else:
        wgt = None

    best_exp, best_err, best_q = None, None, None
    for off in offsets:
        e = (base + off).clamp(-127, 127)
        q = _quantize_elemwise_core(blocked / (2.0 ** e), mbits, ebits, max_norm,
                                    round="nearest", allow_denorm=True, saturate_normals=True)
        q = q * (2.0 ** e)
        d = (blocked - q) ** 2
        if wgt is not None:
            d = d * wgt
        err = d.sum(-1, keepdim=True)
        if best_err is None:
            best_exp, best_err, best_q = e, err, q
        else:
            take = err < best_err
            best_exp = torch.where(take, e, best_exp)
            best_q = torch.where(take, q, best_q)
            best_err = torch.where(take, err, best_err)

    deq = _undo_reshape_to_blocks(best_q, padded_shape, orig_shape, axes)
    return deq, best_exp.squeeze(sax).to(torch.int16), (axes, orig_shape, padded_shape)


def block_exponent(x, elem_format):
    """Shared exponent for one block laid out as rows by block size."""
    _, _, emax, _, _ = _get_format_params(elem_format)
    mx = x.abs().amax(dim=-1)
    e = torch.floor(torch.log2(mx + (mx == 0).float() * 1e-30)) - emax
    return e.clamp(-127, 127)


def quantize_with_exponent(x, elem_format, exp):
    """Quantize onto the format grid using an explicit per row exponent."""
    from mx.elemwise_ops import _quantize_elemwise_core
    ebits, mbits, _, max_norm, _ = _get_format_params(elem_format)
    s = 2.0 ** exp
    q = _quantize_elemwise_core(x / s, mbits, ebits, max_norm, round="nearest",
                                allow_denorm=True, saturate_normals=True)
    return q * s

In [ ]:
%%writefile src/stats.py
# per tensor quantization statistics, accumulated in row chunks
import numpy as np
import torch

from mxfmt import quantize_full, level_table, bits_per_value, block_max_ratio, block_shape_stats, BLOCK, SCALE_BITS, ELEM_BITS


def _entropy(counts):
    p = counts[counts > 0].astype(np.float64)
    p /= p.sum()
    return float(-(p * np.log2(p)).sum())


def tensor_stats(w, elem_format, chunk_rows=4096):
    """Quantize a 2D weight in row chunks and return error, distribution and size stats."""
    n = w.numel()
    tab = level_table(elem_format)
    n_codes = len(tab)
    max_norm = np.nanmax(tab)

    acc = dict(sw2=0.0, se2=0.0, sw4=0.0, maxerr=0.0, n=0, sat=0, satblk=0, nblk=0)
    code_hist = np.zeros(n_codes, dtype=np.int64)
    exp_hist = {}
    n_blocks = 0

    for i in range(0, w.shape[0], chunk_rows):
        c = w[i:i + chunk_rows].float()
        deq, se, codes, _ = quantize_full(c, elem_format)
        d = (c - deq)
        acc["sw2"] += float(c.pow(2).sum())
        acc["se2"] += float(d.pow(2).sum())
        acc["sw4"] += float(c.pow(4).sum())
        acc["maxerr"] = max(acc["maxerr"], float(d.abs().max()))
        acc["n"] += c.numel()
        code_hist += np.bincount(codes.flatten().numpy(), minlength=n_codes)
        for v, cnt in zip(*np.unique(se.numpy(), return_counts=True)):
            exp_hist[int(v)] = exp_hist.get(int(v), 0) + int(cnt)
        n_blocks += se.numel()
        r, mn = block_max_ratio(c, elem_format)
        acc["sat"] += int((r > mn).sum())
        acc["satblk"] += int((r.amax(-1) > mn).sum())
        acc["nblk"] += int(r[..., 0].numel())
        bk, cr = block_shape_stats(c)
        acc["bkurt"] = acc.get("bkurt", 0.0) + float(torch.nansum(bk))
        acc["crest"] = acc.get("crest", 0.0) + float(torch.nansum(cr))
        del c, deq, d, codes, se, r, bk, cr

    vals = tab[np.arange(n_codes)]
    finite = np.isfinite(vals)
    zero_n = int(code_hist[finite & (vals == 0)].sum())
    clip_n = int(code_hist[finite & (np.abs(vals) == max_norm)].sum())

    exp_counts = np.array(list(exp_hist.values()), dtype=np.int64)
    exp_vals = np.array(list(exp_hist.keys()), dtype=np.int64)

    mse = acc["se2"] / acc["n"]
    msw = acc["sw2"] / acc["n"]
    kurt = (acc["sw4"] / acc["n"]) / (msw ** 2) if msw > 0 else float("nan")

    bpv = bits_per_value(elem_format)
    return dict(
        n_params=n,
        rows=w.shape[0], cols=w.shape[1],
        sqnr_db=10 * np.log10(msw / mse) if mse > 0 else float("inf"),
        nmse=mse / msw if msw > 0 else float("nan"),
        max_abs_err=acc["maxerr"],
        rms_weight=float(np.sqrt(msw)),
        kurtosis=kurt,
        block_kurtosis=acc.get("bkurt", 0.0) / max(acc["nblk"], 1),
        block_crest=acc.get("crest", 0.0) / max(acc["nblk"], 1),
        code_entropy=_entropy(code_hist),
        code_entropy_ratio=_entropy(code_hist) / ELEM_BITS[elem_format],
        scale_entropy=_entropy(exp_counts),
        scale_span=int(exp_vals.max() - exp_vals.min()) if len(exp_vals) else 0,
        zero_rate=zero_n / acc["n"],
        top_level_rate=clip_n / acc["n"],
        sat_rate=acc["sat"] / acc["n"],
        sat_block_rate=acc["satblk"] / max(acc["nblk"], 1),
        n_blocks=n_blocks,
        bits_per_value=bpv,
        bytes_bf16=n * 2,
        bytes_mx=int(np.ceil(n * bpv / 8)),
    )

In [ ]:
%%writefile scripts/allocate.py
#!/usr/bin/env python
# pick a format and coarsening strength per tensor under a global bit budget
import argparse, json, os, sys
import numpy as np
import pandas as pd


def frontier(df, thetas):
    """Lagrangian sweep, each theta gives the plan that minimises werr + theta * bytes."""
    out = []
    for th in thetas:
        d = df.assign(cost=df.werr + th * df.bytes)
        pick = d.loc[d.groupby("tensor").cost.idxmin()]
        out.append((th, pick))
    return out


def summarise(pick, total_params):
    bits = pick.bytes.sum() * 8 / total_params
    return dict(bits_per_value=bits, werr=float(pick.werr.sum()), bytes=int(pick.bytes.sum()),
                n_fp8=int((pick.fmt == "mxfp8").sum()), n_fp4=int((pick.fmt == "mxfp4").sum()),
                params_fp8=int(pick.loc[pick.fmt == "mxfp8", "n_params"].sum()))


def uniform(df, fmt, alpha, total_params):
    p = df[(df.fmt == fmt) & (df.alpha == alpha)]
    return summarise(p, total_params), p


def size_greedy(df, budget_bytes, total_params):
    """Naive mixed baseline, upgrade tensors to fp8 from smallest to largest until the budget runs out."""
    f4 = df[(df.fmt == "mxfp4") & (df.alpha == 0)].set_index("tensor")
    f8 = df[(df.fmt == "mxfp8") & (df.alpha == 0)].set_index("tensor")
    cur = f4.bytes.sum()
    chosen = {t: "mxfp4" for t in f4.index}
    for t in f4.sort_values("n_params").index:
        extra = f8.loc[t, "bytes"] - f4.loc[t, "bytes"]
        if cur + extra <= budget_bytes:
            chosen[t] = "mxfp8"
            cur += extra
    rows = [(f8 if v == "mxfp8" else f4).loc[t] for t, v in chosen.items()]
    p = pd.DataFrame(rows).reset_index()
    return summarise(p, total_params), p


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--budgets", default="4.5,5.0,6.0")
    ap.add_argument("--dir", default="results")
    args = ap.parse_args()

    tag = args.model.split("/")[-1]
    df = pd.read_csv(f"{args.dir}/sensitivity_{tag}.csv")
    df = df[~df.role.isin(["embed_tokens", "lm_head"])]
    total_params = int(df[(df.fmt == "mxfp4") & (df.alpha == 0)].n_params.sum())

    ref = {}
    for fmt in ["mxfp4", "mxfp8"]:
        s, _ = uniform(df, fmt, 0.0, total_params)
        ref[f"uniform_{fmt}"] = s
        print(f"uniform {fmt}: {s['bits_per_value']:.3f} b/v  werr {s['werr']:.4g}")

    thetas = np.logspace(-14, -2, 240)
    fr = frontier(df, thetas)
    curve = [summarise(p, total_params) | {"theta": th} for th, p in fr]
    pd.DataFrame(curve).to_csv(f"{args.dir}/frontier_{tag}.csv", index=False)

    plans = {}
    for b in [float(x) for x in args.budgets.split(",")]:
        ok = [(s, p) for s, p in zip(curve, [p for _, p in fr]) if s["bits_per_value"] <= b]
        if not ok:
            print(f"budget {b}: unreachable")
            continue
        s, p = min(ok, key=lambda x: x[0]["werr"])
        nb, _ = size_greedy(df, b * total_params / 8, total_params)
        gain = nb["werr"] / s["werr"]
        print(f"budget {b:.2f}: ours {s['bits_per_value']:.3f} b/v werr {s['werr']:.4g}  "
              f"| size-greedy {nb['bits_per_value']:.3f} b/v werr {nb['werr']:.4g}  | {gain:.2f}x less error")
        plans[str(b)] = dict(summary=s, baseline=nb,
                             plan={r.tensor: dict(fmt=r.fmt, alpha=float(r.alpha)) for r in p.itertuples()})

    with open(f"{args.dir}/plans_{tag}.json", "w") as f:
        json.dump(dict(model=args.model, total_params=total_params, reference=ref, budgets=plans), f, indent=2)
    print(f"wrote {args.dir}/plans_{tag}.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/analyze.py
#!/usr/bin/env python
# build the summary tables and figures from everything under results/
import glob, json, os, sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))

RES = "results"
FIG = "results/figures"
ROLES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens"]
C4, C8 = "#1f77b4", "#d62728"


def load_stats():
    fs = sorted(glob.glob(f"{RES}/stats_*.csv"))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


def model_order(d):
    """Models sorted by parameter count, not alphabetically."""
    n = d[d.fmt == d.fmt.iloc[0]].groupby("model").n_params.sum()
    return list(n.sort_values().index)


def fig_sqnr_uniform(d):
    models = model_order(d)
    cols = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
    fig, axes = plt.subplots(2, 1, figsize=(10, 7))
    for ax, fmt in zip(axes, ["mxfp4", "mxfp8"]):
        s = d[d.fmt == fmt]
        names = [r for r in ROLES if (s.role == r).any()]
        for k, m in enumerate(models):
            g = s[s.model == m]
            xs, ys = [], []
            for i, r in enumerate(names):
                v = g[g.role == r].sqnr_db.values
                xs += list(np.full(len(v), i + (k - 1.5) * 0.17))
                ys += list(v)
            ax.scatter(xs, ys, s=13, alpha=.75, color=cols[k], label=m, edgecolors="none")
        lo, hi = s.sqnr_db.min(), s.sqnr_db.max()
        pad = max(0.05, (hi - lo) * 0.15)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=20, ha="right")
        ax.set_ylabel("SQNR, dB")
        ax.set_title(f"{fmt}, full range {hi - lo:.2f} dB across {len(s)} tensors", fontsize=11)
        ax.grid(alpha=.3, axis="y")
    axes[0].legend(fontsize=8, ncol=4, loc="upper center")
    fig.suptitle("Quantization error is set by the format, not by the layer or the model size")
    fig.tight_layout()
    fig.savefig(f"{FIG}/sqnr_by_role.png", dpi=140)
    plt.close(fig)


def fig_kurtosis(d):
    s = d[d.fmt == "mxfp4"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, col, lab in [(axes[0], "kurtosis", "kurtosis of the whole tensor"),
                         (axes[1], "block_kurtosis", "kurtosis inside a 32 value block")]:
        for m, mk in zip(model_order(s), ["o", "s", "^", "v"]):
            g = s[s.model == m]
            ax.scatter(g[col], g.sqnr_db, s=14, alpha=.6, marker=mk, label=m)
        r = np.corrcoef(s[col], s.sqnr_db)[0, 1]
        ax.set_xlabel(lab)
        ax.set_ylabel("SQNR, dB")
        ax.set_title(f"r = {r:+.2f}")
        ax.grid(alpha=.3)
        ax.legend(fontsize=8)
    axes[0].set_xscale("log")
    fig.suptitle("The usual outlier statistic does not predict MX error, the within block one does")
    fig.tight_layout()
    fig.savefig(f"{FIG}/kurtosis_vs_sqnr.png", dpi=140)
    plt.close(fig)


def fig_memory():
    f = f"{RES}/memory.csv"
    if not os.path.exists(f):
        return
    d = pd.read_csv(f)
    models = list(d[d.fmt == "bf16"].sort_values("bytes").model)
    fig, ax = plt.subplots(figsize=(1.9 * len(models) + 3, 4))
    w, fmts = 0.26, ["bf16", "mxfp8", "mxfp4"]
    for i, fmt in enumerate(fmts):
        g = d[d.fmt == fmt].set_index("model").reindex(models)
        x = np.arange(len(models)) + (i - 1) * w
        ax.bar(x, g.mib / 1024, w, label=fmt, color=["#888", C8, C4][i])
        for xi, v in zip(x, g.mib / 1024):
            ax.text(xi, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models)
    ax.set_ylabel("weights, GiB")
    ax.legend()
    ax.grid(alpha=.3, axis="y")
    ax.set_title("Measured weight footprint")
    fig.tight_layout()
    fig.savefig(f"{FIG}/memory.png", dpi=140)
    plt.close(fig)


def fig_frontier():
    for f in sorted(glob.glob(f"{RES}/frontier_*.csv")):
        tag = os.path.basename(f)[len("frontier_"):-4]
        d = pd.read_csv(f).sort_values("bits_per_value")
        pj = f"{RES}/plans_{tag}.json"
        if not os.path.exists(pj):
            continue
        p = json.load(open(pj))
        fig, ax = plt.subplots(figsize=(6.5, 4.4))
        ax.plot(d.bits_per_value, d.werr, "-", color=C4, label="ours, sensitivity driven")
        for k, s in p["reference"].items():
            ax.scatter([s["bits_per_value"]], [s["werr"]], marker="*", s=170, zorder=5,
                       color=C8 if "fp8" in k else "#444", label=k.replace("_", " "))
        bl = [(v["baseline"]["bits_per_value"], v["baseline"]["werr"]) for v in p["budgets"].values()]
        if bl:
            bl = np.array(sorted(bl))
            ax.plot(bl[:, 0], bl[:, 1], "--o", color="#999", ms=4, label="naive mixed baseline")
        ax.set_yscale("log")
        ax.set_xlabel("bits per value, measured after entropy coding")
        ax.set_ylabel("activation weighted squared error")
        ax.set_title(f"Rate distortion, {tag}")
        ax.grid(alpha=.3)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(f"{FIG}/frontier_{tag}.png", dpi=140)
        plt.close(fig)


BITS = {"bf16": 16.0, "mxfp8": 8.25, "mxfp4": 4.25}


def ppl_table():
    rows = []
    for f in sorted(glob.glob(f"{RES}/ppl_*_kaggle.csv")):
        d = pd.read_csv(f)
        base = {"none": 16.0, "mxfp8": 8.25, "mxfp4": 4.25}
        for r in d.itertuples():
            rows.append(dict(run=f"{r.model}_{r.config}", model=r.model, weights=r.weights,
                             acts=r.acts, errcomp="mxfp4" if "compensation" in str(r.method) else None,
                             search="search" in str(r.method), budget=None,
                             bits=base.get(r.weights, 16.0), ppl=r.ppl, seconds=r.seconds))
    for f in sorted(glob.glob(f"{RES}/ppl/*.json")):
        r = json.load(open(f))
        name = os.path.basename(f)[:-5]
        w = r["weights"] if r["weights"] != "none" else ("bf16" if not r.get("errcomp") else r["errcomp"])
        bits = r.get("bits_per_value") or BITS.get(w, 16.0)
        rows.append(dict(run=name, model=r["model"].split("/")[-1], weights=r["weights"],
                         acts=r["acts"], errcomp=r.get("errcomp"), search=r.get("search", False),
                         budget=r.get("budget"), bits=bits, ppl=r["ppl"], seconds=round(r["seconds"])))
    if not rows:
        return pd.DataFrame()
    d = pd.DataFrame(rows).sort_values(["model", "ppl"])
    d.to_csv(f"{RES}/ppl_summary.csv", index=False)
    return d


def fig_ppl(d):
    """Perplexity against real storage cost, one panel per model."""
    for model, g in d.groupby("model"):
        base = g[(g.weights == "none") & (g.acts == "none") & g.errcomp.isna()]
        if base.empty:
            continue
        fig, ax = plt.subplots(figsize=(7, 4.6))
        ax.axhline(base.ppl.iloc[0], color="#444", ls=":", lw=1)
        ax.text(4.3, base.ppl.iloc[0], " bf16", va="bottom", fontsize=8, color="#444")
        act_w = g.run.str.contains("search_act")
        groups = [
            (g[g.budget.notna()], "bit allocation, rejected", "x", "#999", 55, 3),
            (g[act_w], "exponent search, activation weighted, rejected", "x", "#999", 55, 3),
            (g[(g.acts != "none") & g.errcomp.isna()], "activations quantized", "^", "#9467bd", 70, 4),
            (g[(g.acts == "none") & g.errcomp.isna() & g.search & ~act_w], "exponent search", "s", "#2ca02c", 62, 5),
            (g[g.errcomp.notna()], "error compensation", "D", C8, 72, 6),
            (g[(g.acts == "none") & g.errcomp.isna() & (~g.search) & (g.weights != "none") & g.budget.isna()],
             "MX as normally applied", "o", C4, 110, 7),
        ]
        seen = set()
        for sub, lab, mk, c, sz, z in groups:
            if len(sub) and lab not in seen:
                face = "none" if mk == "o" else c
                ax.scatter(sub.bits, sub.ppl, marker=mk, s=sz, facecolors=face, edgecolors=c,
                           linewidths=1.8, label=lab, zorder=z)
                seen.add(lab)
        ax.set_xlabel("bits per value of the quantized weights")
        ax.set_ylabel("wikitext2 perplexity")
        ax.set_title(f"{model}, lower and left is better")
        ax.grid(alpha=.3)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(f"{FIG}/ppl_vs_bits_{model}.png", dpi=140)
        plt.close(fig)


def eval_table():
    rows = []
    for f in sorted(glob.glob(f"{RES}/eval_*_kaggle.csv")):
        d = pd.read_csv(f)
        for r in d.itertuples():
            rows.append({"run": f"{r.model}_{r.config}", "model": r.model, "weights": r.weights,
                         "acts": r.acts, "budget": None, "seconds": None,
                         "arc_easy/acc": r.arc_easy_acc, "arc_easy/acc_stderr": r.arc_easy_acc_stderr,
                         "piqa/acc": r.piqa_acc, "piqa/acc_stderr": r.piqa_acc_stderr,
                         "winogrande/acc": r.winogrande_acc,
                         "winogrande/acc_stderr": r.winogrande_acc_stderr,
                         "lambada_openai/perplexity": r.lambada_ppl,
                         "lambada_openai/acc": r.lambada_acc,
                         "lambada_openai/acc_stderr": r.lambada_acc_stderr})
    for f in sorted(glob.glob(f"{RES}/eval/*.json")):
        p = json.load(open(f))
        r = {"run": os.path.basename(f)[:-5], "model": p["model"].split("/")[-1],
             "weights": p["weights"], "acts": p["acts"], "budget": p.get("budget"),
             "seconds": round(p["seconds"])}
        for task, m in p["results"].items():
            for k, v in m.items():
                if k.endswith(",none") and not k.startswith("alias") and isinstance(v, float):
                    r[f"{task}/{k.split(',')[0]}"] = round(v, 4)
        rows.append(r)
    if not rows:
        return pd.DataFrame()
    d = pd.DataFrame(rows)
    d.to_csv(f"{RES}/eval_summary.csv", index=False)
    return d


def fig_benchmark(model="Qwen3-4B"):
    """How much of the MXFP4 damage the method gives back, per task."""
    f = f"{RES}/eval_{model}_kaggle.csv"
    if not os.path.exists(f):
        return
    d = pd.read_csv(f).set_index("config")
    need = ["bf16", "w-mxfp4", "errcomp-mxfp4_search"]
    if any(c not in d.index for c in need):
        return
    b, m, o = (d.loc[c] for c in need)
    tasks = [("arc_easy_acc", "arc_easy"), ("piqa_acc", "piqa"),
             ("winogrande_acc", "winogrande"), ("lambada_acc", "lambada acc")]

    fig, ax = plt.subplots(figsize=(8, 4.4))
    x = np.arange(len(tasks))
    w = 0.26
    for k, (row, lab, c) in enumerate([(b, "bf16", "#444"), (m, "MXFP4", C4),
                                       (o, "MXFP4 plus our method", C8)]):
        vals = [row[t] for t, _ in tasks]
        errs = [row[t + "_stderr"] for t, _ in tasks]
        ax.bar(x + (k - 1) * w, vals, w, yerr=errs, capsize=3, label=lab,
               color=c, alpha=.85 if k else .6)
    for i, (t, _) in enumerate(tasks):
        gap = b[t] - m[t]
        if abs(gap) > 1e-9:
            ax.text(i + w, o[t] + 0.02, f"{(o[t]-m[t])/gap*100:.0f}%", ha="center", fontsize=9,
                    color=C8, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([n for _, n in tasks])
    ax.set_ylabel("accuracy")
    ax.set_ylim(0, max(b[t] for t, _ in tasks) * 1.22)
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(alpha=.3, axis="y")
    ax.set_title(f"{model}, share of the MXFP4 loss recovered at the same 4.25 bits")
    fig.tight_layout()
    fig.savefig(f"{FIG}/benchmark_{model}.png", dpi=140)
    plt.close(fig)


def main():
    os.makedirs(FIG, exist_ok=True)
    d = load_stats()
    if len(d):
        fig_sqnr_uniform(d)
        fig_kurtosis(d)
        summary = d.groupby(["model", "fmt"]).agg(
            sqnr_mean=("sqnr_db", "mean"), sqnr_sd=("sqnr_db", "std"),
            sqnr_min=("sqnr_db", "min"), sqnr_max=("sqnr_db", "max"),
            code_entropy=("code_entropy", "mean"), scale_entropy=("scale_entropy", "mean"),
            sat_block=("sat_block_rate", "mean"), params=("n_params", "sum")).round(4)
        summary.to_csv(f"{RES}/stats_summary.csv")
        print(summary.to_string())
    fig_memory()
    fig_frontier()
    p = ppl_table()
    if len(p):
        fig_ppl(p)
        print("\n", p[["run", "bits", "ppl"]].to_string(index=False))
    fig_benchmark()
    e = eval_table()
    if len(e):
        print("\n", e.to_string(index=False))
    print(f"\nfigures in {FIG}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/calibrate.py
#!/usr/bin/env python
# collect per input channel activation norms for every linear, used as importance weights
import argparse, os, sys, time
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from fakequant import target_linears
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--nsamples", type=int, default=64)
    ap.add_argument("--seqlen", type=int, default=1024)
    ap.add_argument("--device", default="mps" if torch.backends.mps.is_available() else "cpu")
    ap.add_argument("--device-map", default=None)
    ap.add_argument("--max-memory", default=None, help="per device cap, e.g. 0=13GiB,1=13GiB,cpu=20GiB")
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--out", default="results/calib")
    args = ap.parse_args()

    tok = AutoTokenizer.from_pretrained(args.model)
    dt = getattr(torch, args.dtype)
    kw = dict(device_map=args.device_map) if args.device_map else {}
    if args.max_memory:
        kw["max_memory"] = {(int(k) if k.isdigit() else k): v
                            for k, v in (p.split("=") for p in args.max_memory.split(","))}
    try:
        model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dt, **kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=dt, **kw)
    if not args.device_map:
        model = model.to(args.device)
    model = model.eval()
    dev = next(model.parameters()).device if args.device_map else args.device

    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(ds["text"])
    ids = tok(text, return_tensors="pt").input_ids[0]

    body = model.model if hasattr(model, "model") else model  # skip lm_head, logits are not needed
    acc, counts = {}, {}
    hooks = []

    def mk(name):
        def hook(mod, args_):
            x = args_[0].detach().float().reshape(-1, args_[0].shape[-1])
            s = x.pow(2).sum(0).cpu()
            acc[name] = s if name not in acc else acc[name] + s
            counts[name] = counts.get(name, 0) + x.shape[0]
        return hook

    for name, m in target_linears(model):
        hooks.append(m.register_forward_pre_hook(mk(name)))

    t0 = time.time()
    step = ids.numel() // args.nsamples
    with torch.no_grad():
        for i in range(args.nsamples):
            chunk = ids[i * step: i * step + args.seqlen].unsqueeze(0).to(dev)
            if chunk.shape[1] < args.seqlen:
                break
            body(chunk)
            if (i + 1) % 8 == 0:
                print(f"{i+1}/{args.nsamples} {time.time()-t0:.0f}s", flush=True)

    for h in hooks:
        h.remove()
    norms = {k: (v / counts[k]).sqrt() for k, v in acc.items()}

    os.makedirs(args.out, exist_ok=True)
    tag = args.model.split("/")[-1]
    torch.save(norms, f"{args.out}/{tag}.pt")
    print(f"wrote {args.out}/{tag}.pt  {len(norms)} linears  {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/memory_report.py
#!/usr/bin/env python
# memory footprint of bf16 vs MX formats, from measured per tensor sizes
import argparse, glob, os, sys
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from mxfmt import bits_per_value, FORMATS, BLOCK, SCALE_BITS, ELEM_BITS

MB = 1024 ** 2


def report(stats_csv, skipped_csv):
    d = pd.read_csv(stats_csv)
    model = d.model.iloc[0]
    kept = pd.read_csv(skipped_csv) if os.path.exists(skipped_csv) else pd.DataFrame(columns=["n_params"])
    n_1d = int(kept.n_params.sum()) if len(kept) else 0

    rows = []
    base = d[d.fmt == d.fmt.iloc[0]]
    n2d = int(base.n_params.sum())
    bf16_bytes = n2d * 2 + n_1d * 2
    rows.append(dict(model=model, fmt="bf16", bits_per_value=16.0, quantized_params=0,
                     bf16_params=n2d + n_1d, bytes=bf16_bytes, mib=bf16_bytes / MB, ratio=1.0))

    for fmt, g in d.groupby("fmt"):
        b = int(g.bytes_mx.sum()) + n_1d * 2
        rows.append(dict(model=model, fmt=fmt, bits_per_value=bits_per_value(FORMATS[fmt]),
                         quantized_params=n2d, bf16_params=n_1d, bytes=b, mib=b / MB,
                         ratio=bf16_bytes / b))
    return pd.DataFrame(rows)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", default="results")
    args = ap.parse_args()

    out = []
    for f in sorted([f for f in glob.glob(f"{args.dir}/stats_*.csv") if "summary" not in f]):
        tag = os.path.basename(f)[len("stats_"):-4]
        out.append(report(f, f"{args.dir}/skipped_{tag}.csv"))
    df = pd.concat(out, ignore_index=True)
    df.to_csv(f"{args.dir}/memory.csv", index=False)

    print(f"block size {BLOCK}, scale {SCALE_BITS} bits shared per block")
    for f, ef in FORMATS.items():
        if f in df.fmt.values:
            print(f"  {f:7s} {ELEM_BITS[ef]} + {SCALE_BITS}/{BLOCK} = {bits_per_value(ef)} bits per value, "
                  f"{16/bits_per_value(ef):.3f}x vs bf16 on quantized tensors")
    print()
    show = df.copy()
    show["MiB"] = show.mib.round(1)
    show["GiB"] = (show.mib / 1024).round(3)
    print(show[["model", "fmt", "bits_per_value", "bytes", "MiB", "GiB", "ratio"]].to_string(index=False))
    print(f"\nwrote {args.dir}/memory.csv")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/ppl.py
#!/usr/bin/env python
# direct perplexity on wikitext2, more sensitive than accuracy on small samples
import argparse, json, os, sys, time
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from fakequant import build_model
from datasets import load_dataset


@torch.no_grad()
def perplexity(model, tok, device, seqlen=1024, nwin=32, chunk=128):
    """Sliced cross entropy, the full logits tensor does not fit next to a 150k vocabulary."""
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    ids = tok("\n\n".join(ds["text"]), return_tensors="pt").input_ids[0]
    nll, ntok = 0.0, 0
    for i in range(min(nwin, ids.numel() // seqlen)):
        x = ids[i * seqlen:(i + 1) * seqlen].unsqueeze(0).to(device)
        body = model.model if hasattr(model, "model") else model
        h = body(x)[0][0, :-1]
        tgt = x[0, 1:]
        for j in range(0, h.shape[0], chunk):
            lg = model.lm_head(h[j:j + chunk]).float()
            nll += torch.nn.functional.cross_entropy(lg, tgt[j:j + chunk], reduction="sum").item()
            del lg
        ntok += tgt.numel()
        del h, x
    return float(torch.exp(torch.tensor(nll / ntok)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--weights", default="none")
    ap.add_argument("--acts", default="none")
    ap.add_argument("--plan", default=None)
    ap.add_argument("--budget", default=None)
    ap.add_argument("--calib", default=None)
    ap.add_argument("--search", action="store_true")
    ap.add_argument("--errcomp", default=None)
    ap.add_argument("--calib-source", default="wikitext", choices=["wikitext", "c4"])
    ap.add_argument("--errcomp-group", type=int, default=2)
    ap.add_argument("--errcomp-nsamples", type=int, default=16)
    ap.add_argument("--nwin", type=int, default=32)
    ap.add_argument("--seqlen", type=int, default=1024)
    ap.add_argument("--device", default="mps" if torch.backends.mps.is_available() else "cpu")
    ap.add_argument("--device-map", default=None, help="auto spreads the model over every visible GPU")
    ap.add_argument("--max-memory", default=None)
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--tag", default=None)
    ap.add_argument("--out", default="results/ppl")
    args = ap.parse_args()

    t0 = time.time()
    model, tok, act, info = build_model(args.model, args.weights, args.acts, args.device,
                                        getattr(torch, args.dtype), False, args.plan, args.budget,
                                        args.calib, args.search, args.device_map, args.max_memory)
    if args.errcomp:
        from errcomp import apply_compensated, calib_tokens
        ids = calib_tokens(tok, args.calib_source)
        dev0 = next(model.parameters()).device if args.device_map else args.device
        apply_compensated(model, ids, args.errcomp, dev0, group=args.errcomp_group,
                   nsamples=args.errcomp_nsamples, seqlen=args.seqlen, search=args.search)
    dev = next(model.parameters()).device if args.device_map else args.device
    ppl = perplexity(model, tok, dev, args.seqlen, args.nwin)
    tag = args.tag or f"{args.model.split('/')[-1]}_w-{args.weights}_a-{args.acts}"
    os.makedirs(args.out, exist_ok=True)
    rec = dict(model=args.model, weights=args.weights, acts=args.acts, plan=args.plan, search=args.search,
               errcomp=args.errcomp,
               budget=args.budget, nwin=args.nwin, seqlen=args.seqlen, ppl=ppl,
               dtype=args.dtype, device_map=args.device_map,
               bits_per_value=info.get("bits_per_value"), seconds=time.time() - t0)
    with open(f"{args.out}/{tag}.json", "w") as f:
        json.dump(rec, f, indent=2)
    print(f"{tag}: ppl {ppl:.4f}  bits {info.get('bits_per_value')}  {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/quantize_model.py
#!/usr/bin/env python
# quantize every 2D weight of a checkpoint and dump per tensor stats
import argparse, json, os, sys, time
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from modelio import local_path, iter_tensors, classify, layer_of, tensor_index
from stats import tensor_stats
from mxfmt import FORMATS


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--formats", default="mxfp4,mxfp8")
    ap.add_argument("--out", default="results")
    args = ap.parse_args()

    path = local_path(args.model)
    tag = args.model.split("/")[-1]
    rows = []
    t0 = time.time()

    tied = json.load(open(os.path.join(path, "config.json"))).get("tie_word_embeddings", False)

    for name, w in iter_tensors(path):
        if tied and name == "lm_head.weight":
            print(f"skip {name}, tied to embed_tokens")
            continue
        for fmt in args.formats.split(","):
            s = tensor_stats(w, FORMATS[fmt])
            s.update(model=tag, tensor=name, fmt=fmt, role=classify(name), layer=layer_of(name),
                     dtype=str(w.dtype).replace("torch.", ""))
            rows.append(s)
        print(f"{time.time()-t0:7.1f}s {name} {tuple(w.shape)}", flush=True)

    # 1D tensors stay in bf16, count them so the memory budget is honest
    idx = tensor_index(path)
    skipped = {k: v for k, v in idx.items() if len(v[0]) != 2}
    os.makedirs(args.out, exist_ok=True)
    df = pd.DataFrame(rows)
    df.to_csv(f"{args.out}/stats_{tag}.csv", index=False)
    pd.DataFrame([{"tensor": k, "shape": str(v[0]), "dtype": v[1],
                   "n_params": int(v[0][0]) if v[0] else 0} for k, v in skipped.items()]).to_csv(
        f"{args.out}/skipped_{tag}.csv", index=False)
    print(f"wrote {args.out}/stats_{tag}.csv  {len(df)} rows  {time.time()-t0:.1f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/run_eval.py
#!/usr/bin/env python
# lm-eval on a model with optional MX fake quantization of weights and activations
import argparse, json, os, sys, time
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from fakequant import quantize_weights, ActQuant, apply_plan
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer


def build(model_id, wfmt, afmt, device, dtype, lm_head, plan_file=None, budget=None, calib=None,
          search=False, device_map=None, max_memory=None):
    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(device_map=device_map) if device_map else {}
    if max_memory:
        kw["max_memory"] = {(int(k) if k.isdigit() else k): v
                            for k, v in (q.split("=") for q in max_memory.split(","))}
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, **kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, **kw)
    if not device_map:
        model = model.to(device)
    model = model.eval()
    act = None
    if plan_file:
        with open(plan_file) as f:
            plans = json.load(f)
        plan = plans["budgets"][budget]["plan"]
        norms = torch.load(calib) if calib else None
        t = time.time()
        info = apply_plan(model, plan, norms)
        print(f"applied plan at budget {budget}: {info['tensors']} tensors, "
              f"{info['bits_per_value']:.3f} bits per value, {time.time()-t:.0f}s", flush=True)
    elif wfmt != "none":
        t = time.time()
        n = quantize_weights(model, wfmt, lm_head, search=search)
        how = " with exponent search" if search else ""
        print(f"quantized {n/1e6:.0f}M weights to {wfmt}{how} in {time.time()-t:.0f}s", flush=True)
    if afmt != "none":
        act = ActQuant(model, afmt, lm_head)
        print(f"activations quantized to {afmt}", flush=True)
    return model, tok, act


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--weights", default="none")
    ap.add_argument("--acts", default="none")
    ap.add_argument("--tasks", default="wikitext")
    ap.add_argument("--limit", type=float, default=None)
    ap.add_argument("--batch-size", default="4")
    ap.add_argument("--max-length", type=int, default=2048)
    ap.add_argument("--device", default="mps" if torch.backends.mps.is_available() else "cpu")
    ap.add_argument("--device-map", default=None, help="auto spreads the model over every visible GPU")
    ap.add_argument("--max-memory", default=None, help="per device cap, e.g. 0=13GiB,1=13GiB,cpu=20GiB")
    ap.add_argument("--dtype", default="bfloat16")
    ap.add_argument("--quant-lm-head", action="store_true")
    ap.add_argument("--errcomp", default=None)
    ap.add_argument("--calib-source", default="wikitext", choices=["wikitext", "c4"])
    ap.add_argument("--search", action="store_true")
    ap.add_argument("--plan", default=None)
    ap.add_argument("--budget", default=None)
    ap.add_argument("--calib", default=None)
    ap.add_argument("--tag", default=None)
    ap.add_argument("--out", default="results/eval")
    args = ap.parse_args()

    dtype = getattr(torch, args.dtype)
    model, tok, act = build(args.model, args.weights, args.acts, args.device, dtype,
                            args.quant_lm_head, args.plan, args.budget, args.calib, args.search,
                            args.device_map, args.max_memory)
    if args.errcomp:
        from errcomp import apply_compensated, calib_tokens
        ids = calib_tokens(tok, args.calib_source)
        t = time.time()
        dev = next(model.parameters()).device if args.device_map else args.device
        apply_compensated(model, ids, args.errcomp, dev, search=args.search)
        print(f"error compensation in {args.errcomp} done in {time.time()-t:.0f}s", flush=True)

    hf_kw = dict(pretrained=model, tokenizer=tok, batch_size=args.batch_size,
                 max_length=args.max_length)
    if not args.device_map:
        hf_kw["device"] = args.device
    lm = HFLM(**hf_kw)

    t0 = time.time()
    res = lm_eval.simple_evaluate(model=lm, tasks=args.tasks.split(","), limit=args.limit, verbosity="WARNING")
    dt = time.time() - t0

    tag = args.tag or f"{args.model.split('/')[-1]}_w-{args.weights}_a-{args.acts}"
    os.makedirs(args.out, exist_ok=True)
    payload = dict(model=args.model, weights=args.weights, acts=args.acts, tasks=args.tasks,
                   plan=args.plan, budget=args.budget, errcomp=args.errcomp, search=args.search, calib_source=args.calib_source,
                   limit=args.limit, dtype=args.dtype, device=args.device, seconds=dt,
                   max_length=args.max_length, device_map=args.device_map,
                   results=res["results"], n_samples={k: v for k, v in res.get("n-samples", {}).items()})
    with open(f"{args.out}/{tag}.json", "w") as f:
        json.dump(payload, f, indent=2, default=str)
    for task, r in res["results"].items():
        print(task, {k: round(v, 4) for k, v in r.items() if isinstance(v, float)})
    print(f"done in {dt:.0f}s -> {args.out}/{tag}.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/sensitivity.py
#!/usr/bin/env python
# per tensor cost and activation weighted error for each format and coarsening strength
import argparse, os, sys, time
import pandas as pd
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from modelio import local_path, iter_tensors, classify, layer_of
from mxfmt import quantize_full, dequantize_codes, FORMATS
from compress import compress_tensor


def weighted_err(w, rec, a):
    """Sum over the tensor of (dw * activation norm)^2, a proxy for output perturbation."""
    d = (w - rec).float()
    if a is not None:
        d = d * a.view(1, -1)
    return float(d.pow(2).sum())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--alphas", default="0,0.5,2,5")
    ap.add_argument("--calib", default="results/calib")
    ap.add_argument("--out", default="results")
    args = ap.parse_args()

    tag = args.model.split("/")[-1]
    path = local_path(args.model)
    if os.path.exists(f"{args.out}/sensitivity_{tag}.csv"):
        os.remove(f"{args.out}/sensitivity_{tag}.csv")
    norms = torch.load(f"{args.calib}/{tag}.pt")
    key = {k + ".weight": v for k, v in norms.items()}
    alphas = [float(x) for x in args.alphas.split(",")]

    out_csv = f"{args.out}/sensitivity_{tag}.csv"
    os.makedirs(args.out, exist_ok=True)
    rows, t0, wrote_header = [], time.time(), False
    for name, w in iter_tensors(path):
        if classify(name) in ("embed_tokens", "lm_head"):
            continue  # not quantized in the eval path, handled in the memory report
        a = key.get(name)
        w = w.float()
        for fmt, ef in [("mxfp4", FORMATS["mxfp4"]), ("mxfp8", FORMATS["mxfp8"])]:
            deq, se, codes, meta = quantize_full(w, ef)
            ref = (w - deq).pow(2).mean().item()
            base_werr = weighted_err(w, deq, a)
            for al in alphas:
                r = compress_tensor(codes, se, ef, act_norm=a, alpha=al, ref_mse=ref)
                rec = dequantize_codes(r["codes"], se, ef, meta) if al > 0 else deq
                rows.append(dict(model=tag, tensor=name, role=classify(name), layer=layer_of(name),
                                 fmt=fmt, alpha=al, n_params=w.numel(),
                                 bits_per_value=r["bits_per_value"], bits_codes=r["bits_codes"],
                                 bits_scales=r["bits_scales"], bytes=r["bytes_codes"] + r["bytes_scales"],
                                 changed_frac=r["changed_frac"], has_calib=a is not None,
                                 werr=weighted_err(w, rec, a), werr_mxonly=base_werr,
                                 mse=(w - rec).pow(2).mean().item(), mse_mxonly=ref))
            del deq, codes, se
        pd.DataFrame(rows[-2 * len(alphas):]).to_csv(out_csv, mode="a", header=not wrote_header, index=False)
        wrote_header = True
        print(f"{time.time()-t0:7.1f}s {name}", flush=True)

    print(f"wrote {out_csv}  {len(rows)} rows")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/signal_power.py
#!/usr/bin/env python
# per tensor signal power sum (w * activation norm)^2, used to normalise sensitivity
import argparse, os, sys
import pandas as pd
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from modelio import local_path, iter_tensors, classify


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--calib", default="results/calib")
    ap.add_argument("--out", default="results")
    args = ap.parse_args()

    tag = args.model.split("/")[-1]
    norms = torch.load(f"{args.calib}/{tag}.pt")
    key = {k + ".weight": v for k, v in norms.items()}
    rows = []
    for name, w in iter_tensors(local_path(args.model)):
        if classify(name) in ("embed_tokens", "lm_head"):
            continue
        a = key.get(name)
        w = w.float()
        p = (w * a.view(1, -1)).pow(2).sum().item() if a is not None else w.pow(2).sum().item()
        rows.append(dict(tensor=name, wnorm=p, w2=w.pow(2).sum().item()))
    pd.DataFrame(rows).to_csv(f"{args.out}/signal_{tag}.csv", index=False)
    print(f"wrote {args.out}/signal_{tag}.csv  {len(rows)} tensors")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/test_roundtrip.py
#!/usr/bin/env python
# compress every tensor, decompress it, and check the model comes back bit for bit
import argparse, os, sys, time
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from modelio import local_path, iter_tensors, classify
from mxfmt import quantize_full, dequantize_codes, FORMATS
from compress import compress_tensor, decompress_tensor


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--formats", default="mxfp4,mxfp8")
    ap.add_argument("--limit", type=int, default=0, help="stop after this many tensors")
    ap.add_argument("--chunk-rows", type=int, default=2048)
    args = ap.parse_args()

    path = local_path(args.model)
    t0, bad, n_t, n_val, raw, comp = time.time(), 0, 0, 0, 0, 0

    for name, w in iter_tensors(path):
        if classify(name) == "lm_head":
            continue
        for fmt in args.formats.split(","):
            ef = FORMATS[fmt]
            for i in range(0, w.shape[0], args.chunk_rows):
                c = w[i:i + args.chunk_rows].float()
                deq, se, codes, meta = quantize_full(c, ef)
                r = compress_tensor(codes, se, ef, alpha=0.0)
                codes2, se2 = decompress_tensor(r)

                if not torch.equal(codes, codes2):
                    print(f"  codes differ: {name} {fmt} rows {i}")
                    bad += 1
                elif not torch.equal(se, se2):
                    print(f"  exponents differ: {name} {fmt} rows {i}")
                    bad += 1
                else:
                    rebuilt = dequantize_codes(codes2, se2, ef, meta)
                    if not torch.equal(rebuilt, deq):
                        print(f"  tensor differs: {name} {fmt} rows {i}")
                        bad += 1
                n_val += c.numel()
                raw += c.numel() * 2
                comp += r["bytes_codes"] + r["bytes_scales"]
                del c, deq, se, codes, r, codes2, se2
            n_t += 1
        if args.limit and n_t >= args.limit * len(args.formats.split(",")):
            break

    print(f"\n{args.model}")
    print(f"  tensors checked   {n_t}")
    print(f"  values            {n_val:,}")
    print(f"  mismatches        {bad}")
    print(f"  bf16 bytes        {raw:,}")
    print(f"  compressed bytes  {comp:,}")
    print(f"  ratio             {raw / comp:.3f}x")
    print(f"  bits per value    {comp * 8 / n_val:.3f}")
    print(f"  {time.time()-t0:.0f}s")
    return 1 if bad else 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile scripts/verify_packing.py
#!/usr/bin/env python
# pack a quantized model to disk and compare the real file size against the formula
import argparse, json, os, sys, time
import numpy as np
import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "src"))
from modelio import local_path, iter_tensors, classify
from mxfmt import quantize_full, dequantize_codes, FORMATS, ELEM_BITS, SCALE_BITS, BLOCK, bits_per_value
from compress import pack_codes


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("model")
    ap.add_argument("--fmt", default="mxfp4")
    ap.add_argument("--out", default=None)
    ap.add_argument("--check-roundtrip", action="store_true")
    args = ap.parse_args()

    tag = args.model.split("/")[-1]
    ef = FORMATS[args.fmt]
    bits = ELEM_BITS[ef]
    out = args.out or f"/tmp/{tag}_{args.fmt}.bin"
    path = local_path(args.model)

    tied = json.load(open(os.path.join(path, "config.json"))).get("tie_word_embeddings", False)
    n_total, n_blocks, worst = 0, 0, 0.0
    t0 = time.time()
    with open(out, "wb") as f:
        for name, w in iter_tensors(path):
            if tied and classify(name) == "lm_head":
                continue
            for i in range(0, w.shape[0], 2048):
                c = w[i:i + 2048].float()
                deq, se, codes, meta = quantize_full(c, ef)
                f.write(pack_codes(codes.numpy(), bits))
                f.write(se.numpy().astype(np.int8).tobytes())
                n_total += c.numel()
                n_blocks += se.numel()
                if args.check_roundtrip:
                    rt = dequantize_codes(codes, se, ef, meta)
                    worst = max(worst, float((rt - deq).abs().max()))
                del c, deq, se, codes
            print(f"  {name}", flush=True)

    real = os.path.getsize(out)
    formula = int(np.ceil(n_total * bits / 8)) + int(np.ceil(n_blocks * SCALE_BITS / 8))
    print(f"\n{tag} {args.fmt}")
    print(f"  values          {n_total}")
    print(f"  blocks          {n_blocks}  ({n_total / n_blocks:.2f} values per block)")
    print(f"  file on disk    {real} bytes")
    print(f"  formula         {formula} bytes  ({bits} + {SCALE_BITS}/{BLOCK} = {bits_per_value(ef)} bits per value)")
    print(f"  difference      {real - formula} bytes")
    print(f"  measured        {real * 8 / n_total:.4f} bits per value")
    if args.check_roundtrip:
        print(f"  max round trip error {worst}")
    print(f"  {time.time()-t0:.0f}s, file kept at {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/fetch_models.sh
#!/bin/bash
# download checkpoints in order of priority
export HF_HUB_ENABLE_HF_TRANSFER=1
for m in Qwen/Qwen3-1.7B Qwen/Qwen3-0.6B Qwen/Qwen3-4B Qwen/Qwen3-14B; do
  echo "=== $m $(date +%T)"
  ./venv/bin/hf download "$m" || echo "FAILED $m"
done
echo "=== all done $(date +%T)"

In [ ]:
%%writefile scripts/ppl_matrix.sh
#!/bin/bash
# perplexity across all quantization configurations for one model
set -u
M=${1:-Qwen/Qwen3-0.6B}
TAG=$(basename $M)
DEV=${DEV:-mps}
DEVMAP=${DEVMAP:-}
DTYPE=${DTYPE:-bfloat16}
EXTRA=""
MAXMEM=${MAXMEM:-}
[ -n "$DEVMAP" ] && EXTRA="--device-map $DEVMAP"
[ -n "$MAXMEM" ] && EXTRA="$EXTRA --max-memory $MAXMEM"
NWIN=${NWIN:-32}
SEQLEN=${SEQLEN:-1024}
# use the local venv when there is one, plain python on Colab or Kaggle
if [ -z "${PY:-}" ]; then
  if [ -x ./venv/bin/python ]; then PY=./venv/bin/python; else PY=python; fi
fi
# print the result line on success, the tail of the output on failure
run() {
  out=$($PY -u scripts/ppl.py $M --device $DEV --nwin $NWIN --seqlen $SEQLEN --dtype $DTYPE $EXTRA "$@" 2>&1)
  if echo "$out" | grep -aqE "^[A-Za-z0-9_.-]+: ppl "; then
    echo "$out" | grep -aE "^[A-Za-z0-9_.-]+: ppl "
  else
    echo "FAILED: $*"
    echo "$out" | tr '\r' '\n' | grep -av "it/s\]$" | tail -6
  fi
}

# ONLYERRCOMP skips straight to the slow compensated configuration
if [ -n "${ONLYERRCOMP:-}" ]; then
  run --errcomp mxfp4 --search --tag ${TAG}_errcomp-mxfp4_search
  exit 0
fi

# ordered by priority, the task statement asks for weights and activations first
run --tag ${TAG}_bf16
run --weights mxfp8 --tag ${TAG}_w-mxfp8
run --weights mxfp4 --tag ${TAG}_w-mxfp4
[ -n "${MIN:-}" ] && { echo "### minimal set done $(date +%T)"; exit 0; }
run --weights mxfp8 --acts mxfp8 --tag ${TAG}_w-mxfp8_a-mxfp8
run --weights mxfp4 --acts mxfp8 --tag ${TAG}_w-mxfp4_a-mxfp8
[ -z "${NOERRCOMP:-}" ] && run --errcomp mxfp4 --search --tag ${TAG}_errcomp-mxfp4_search

# CORE=1 stops here, everything below is extra
[ -n "${CORE:-}" ] && exit 0

run --acts mxfp8 --tag ${TAG}_a-mxfp8
run --acts mxfp4 --tag ${TAG}_a-mxfp4
run --weights mxfp4 --search --tag ${TAG}_w-mxfp4_search
run --weights mxfp8 --search --tag ${TAG}_w-mxfp8_search
run --errcomp mxfp4 --tag ${TAG}_errcomp-mxfp4
if [ -f results/plans_${TAG}.json ]; then
  for b in 4.2 5.0; do
    run --plan results/plans_${TAG}.json --budget $b --calib results/calib/${TAG}.pt --tag ${TAG}_ours-$b
  done
fi

In [ ]:
%%writefile scripts/run_matrix.sh
#!/bin/bash
# full evaluation matrix for one model
set -u
M=${1:-Qwen/Qwen3-0.6B}
TAG=$(basename $M)
TASKS=${TASKS:-wikitext,arc_easy,piqa,winogrande,lambada_openai}
DEV=${DEV:-mps}
DEVMAP=${DEVMAP:-}
DTYPE=${DTYPE:-bfloat16}
EXTRA=""
MAXMEM=${MAXMEM:-}
[ -n "$DEVMAP" ] && EXTRA="--device-map $DEVMAP"
[ -n "$MAXMEM" ] && EXTRA="$EXTRA --max-memory $MAXMEM"
BS=${BS:-4}
MAXLEN=${MAXLEN:-2048}
# use the local venv when there is one, plain python on Colab or Kaggle
if [ -z "${PY:-}" ]; then
  if [ -x ./venv/bin/python ]; then PY=./venv/bin/python; else PY=python; fi
fi
LIMIT=${LIMIT:-}
LIMARG=""; [ -n "$LIMIT" ] && LIMARG="--limit $LIMIT"
run() {
  echo "### $* $(date +%T)"
  out=$($PY -u scripts/run_eval.py $M --tasks $TASKS --device $DEV --batch-size $BS \
        --max-length $MAXLEN --dtype $DTYPE $EXTRA $LIMARG "$@" 2>&1)
  if echo "$out" | grep -aqE "^[a-z_0-9]+ \{"; then
    echo "$out" | grep -aE "^[a-z_0-9]+ \{"
  else
    echo "FAILED: $*"
    echo "$out" | tr '\r' '\n' | grep -av "it/s\]$" | tail -8
  fi
}

# ONLYERRCOMP skips straight to the slow compensated configuration
if [ -n "${ONLYERRCOMP:-}" ]; then
  run --errcomp mxfp4 --search --tag ${TAG}_errcomp-mxfp4_search
  exit 0
fi

# ordered by priority, the task statement asks for weights and activations first
run --tag ${TAG}_bf16
run --weights mxfp8 --tag ${TAG}_w-mxfp8
run --weights mxfp4 --tag ${TAG}_w-mxfp4
[ -n "${MIN:-}" ] && { echo "### minimal set done $(date +%T)"; exit 0; }
run --weights mxfp8 --acts mxfp8 --tag ${TAG}_w-mxfp8_a-mxfp8
run --weights mxfp4 --acts mxfp8 --tag ${TAG}_w-mxfp4_a-mxfp8
[ -z "${NOERRCOMP:-}" ] && run --errcomp mxfp4 --search --tag ${TAG}_errcomp-mxfp4_search

# CORE=1 stops here, everything below is extra
if [ -z "${CORE:-}" ]; then
  [ -f results/plans_${TAG}.json ] && run --plan results/plans_${TAG}.json --budget 4.2 --calib results/calib/${TAG}.pt --tag ${TAG}_ours-4.2
fi
echo "### matrix done $(date +%T)"

## Calibration

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python scripts/calibrate.py Qwen/Qwen3-14B --nsamples 64 --seqlen 512 --device cuda --dtype float16 --device-map auto --max-memory 0=13GiB,1=13GiB,cpu=24GiB

## Perplexity

In [ ]:
!MIN=1 NOERRCOMP=1 CORE=1 DEV=cuda DTYPE=float16 DEVMAP=auto MAXMEM=0=13GiB,1=13GiB,cpu=24GiB PYTORCH_ALLOC_CONF=expandable_segments:True NWIN=32 SEQLEN=512 bash scripts/ppl_matrix.sh Qwen/Qwen3-14B

In [ ]:
import os
for d in ['results/ppl', 'results/eval']:
    fs = sorted(os.listdir(d)) if os.path.isdir(d) else []
    print(d, len(fs), 'files'); [print('  ', f) for f in fs]

In [ ]:
!cd /kaggle/working && zip -qr results_14b.zip results && ls -la results_14b.zip

## lm-eval
First, because the benchmark is what the task statement asks for.

In [ ]:
!TASKS=arc_easy,lambada_openai MIN=1 NOERRCOMP=1 CORE=1 DEV=cuda DTYPE=float16 DEVMAP=auto MAXMEM=0=13GiB,1=13GiB,cpu=24GiB PYTORCH_ALLOC_CONF=expandable_segments:True BS=8 MAXLEN=512 \
  bash scripts/run_matrix.sh Qwen/Qwen3-14B

In [ ]:
!cd /kaggle/working && zip -qr results_14b.zip results && ls -la results_14b.zip

# Qwen3-14B, the method from this project

Everything above is saved before this starts. The Hessian solve at this size is the
most expensive step in the project, so it runs last and has its own archive.

## lm-eval with error compensation

In [ ]:
!TASKS=arc_easy,lambada_openai ONLYERRCOMP=1 DEV=cuda DTYPE=float16 DEVMAP=auto MAXMEM=0=13GiB,1=13GiB,cpu=24GiB PYTORCH_ALLOC_CONF=expandable_segments:True BS=8 MAXLEN=512 \
  bash scripts/run_matrix.sh Qwen/Qwen3-14B

## Perplexity with error compensation

In [ ]:
!ONLYERRCOMP=1 DEV=cuda DTYPE=float16 DEVMAP=auto MAXMEM=0=13GiB,1=13GiB,cpu=24GiB PYTORCH_ALLOC_CONF=expandable_segments:True NWIN=32 SEQLEN=512 \
  bash scripts/ppl_matrix.sh Qwen/Qwen3-14B

In [ ]:
import os
for d in ['results/ppl', 'results/eval']:
    fs = sorted(os.listdir(d)) if os.path.isdir(d) else []
    print(d, len(fs), 'files'); [print('  ', f) for f in fs]

In [ ]:
!cd /kaggle/working && zip -qr results_14b.zip results && ls -la results_14b.zip